## 01a Experiment 1a: Category Membership - Items Task

### 0 Setup

In [1]:
import pandas as pd
import csv
from google import genai
from google.genai import types
import time
import random

### 1 Functions

In [2]:
# Prompt Model to answer Questionnaire

def collect_output(client, prompt):
    response = client.models.generate_content(
        model="gemini-3-pro-preview",
        contents=prompt)
    return response

### A. Items Task

In [3]:
# CATEGORIES
categories = ["Fruit","Vehicle","Weapon","Vegetable", "Carpenter's tool", "Bird", "Sport", "Toy", "Clothing"] #from rosch1975a

# + "Furniture" category, but "furniture" excluded here, because using results from prompts experiment with prompt 6

In [ ]:
# Collect Answers from Model

items_reps = 5 #30

# retrieve API 
with open('', 'r') as file:
    api = file.read()     
client = genai.Client(api_key=api)

prompt_outputs = pd.DataFrame(columns=['model', 'prompt', 'prompt_id', 'runtime', 'category', 'output'])
model_output = pd.DataFrame()

for rep in range(items_reps): # max rep == 30

        rep_categories = categories.copy() 
        random.shuffle(rep_categories)
        
        for i in range(len(rep_categories)):

            current_prompt = f"""
            The procedure will be as follows: First you will be given the name or description of a category. Then you will write as many items included in that category as you can, in whatever order they happen to occur to you. The category is: "{rep_categories[i]}".
            Do not add any further details, only write the items that answer the instruction.
            Provide your answer in plain text as a comma-separated list of nouns.
            """
            
            start_time = time.time() 
            output = collect_output(client, current_prompt)
            end_time = time.time() 
            runtime = end_time - start_time
            model_output = pd.DataFrame({
                'model': 'gemini-3-pro',
                'prompt': [current_prompt],
                'prompt_id': [6],
                'runtime': [runtime],
                'category': [rep_categories[i]],
                'output': [output]
            })
            
            prompt_outputs = pd.concat([prompt_outputs, model_output], ignore_index=True)

prompt_outputs.head()

C:\Users\AS\AppData\Local\Temp\ipykernel_16256\3635328170.py:39: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  prompt_outputs = pd.concat([prompt_outputs, model_output], ignore_index=True)


,model,prompt,prompt_id,runtime,category,output
0,gemini-3-pro,\n The procedure will be as follows...,6,87.558914,Vehicle,sdk_http_response=HttpResponse(\n headers=<di...
1,gemini-3-pro,\n The procedure will be as follows...,6,67.749557,Weapon,sdk_http_response=HttpResponse(\n headers=<di...
2,gemini-3-pro,\n The procedure will be as follows...,6,64.604481,Toy,sdk_http_response=HttpResponse(\n headers=<di...
3,gemini-3-pro,\n The procedure will be as follows...,6,64.333744,Fruit,sdk_http_response=HttpResponse(\n headers=<di...
4,gemini-3-pro,\n The procedure will be as follows...,6,79.516820,Bird,sdk_http_response=HttpResponse(\n headers=<di...


In [10]:
prompt_outputs

,model,prompt,prompt_id,runtime,category,output
0,gemini-3-pro,\n The procedure will be as follows...,6,87.558914,Vehicle,"car, truck, bus, bicycle, motorcycle, van, sco..."
1,gemini-3-pro,\n The procedure will be as follows...,6,67.749557,Weapon,"sword, knife, gun, pistol, rifle, shotgun, dag..."
2,gemini-3-pro,\n The procedure will be as follows...,6,64.604481,Toy,"doll, action figure, teddy bear, stuffed anima..."
3,gemini-3-pro,\n The procedure will be as follows...,6,64.333744,Fruit,"apple, banana, orange, pear, grape, strawberry..."
4,gemini-3-pro,\n The procedure will be as follows...,6,79.516820,Bird,"robin, sparrow, eagle, hawk, owl, penguin, ost..."
5,gemini-3-pro,\n The procedure will be as follows...,6,48.091189,Sport,"soccer, basketball, tennis, baseball, american..."
6,gemini-3-pro,\n The procedure will be as follows...,6,49.131781,Carpenter's tool,"Hammer, saw, drill, tape measure, chisel, plan..."
7,gemini-3-pro,\n The procedure will be as follows...,6,47.765399,Clothing,"shirt, pants, dress, skirt, shorts, socks, sho..."
8,gemini-3-pro,\n The procedure will be as follows...,6,38.017884,Vegetable,"Carrot, potato, broccoli, cauliflower, spinach..."
9,gemini-3-pro,\n The procedure will be as follows...,6,73.123747,Clothing,"shirt, pants, dress, skirt, blouse, sweater, j..."


In [ ]:
# Cleaning Model Output

text = ""
response = prompt_outputs['output'][0]

def extract_text(row):
    text = ""
    for candidate in row['output'].candidates:
        for part in candidate.content.parts:
            text += part.text + "\n"  
    return text.strip()  

prompt_outputs['output'] = prompt_outputs.apply(extract_text, axis=1)

In [12]:
prompt_outputs.head()

,model,prompt,prompt_id,runtime,category,output
0,gemini-3-pro,\n The procedure will be as follows...,6,87.558914,Vehicle,"car, truck, bus, bicycle, motorcycle, van, sco..."
1,gemini-3-pro,\n The procedure will be as follows...,6,67.749557,Weapon,"sword, knife, gun, pistol, rifle, shotgun, dag..."
2,gemini-3-pro,\n The procedure will be as follows...,6,64.604481,Toy,"doll, action figure, teddy bear, stuffed anima..."
3,gemini-3-pro,\n The procedure will be as follows...,6,64.333744,Fruit,"apple, banana, orange, pear, grape, strawberry..."
4,gemini-3-pro,\n The procedure will be as follows...,6,79.516820,Bird,"robin, sparrow, eagle, hawk, owl, penguin, ost..."


In [13]:
len(prompt_outputs)

45

In [14]:
# take away NaN Answers
print("Number of rows:", len(prompt_outputs),"\n")
prompt_outputs.dropna(inplace=True)
print("Number of rows after NA filtering:", len(prompt_outputs))

Number of rows: 45 

Number of rows after NA filtering: 45


In [15]:
prompt_outputs.to_csv(f'01_catmemexp_itemstask_allmodels_raw.csv',index=False, quoting=csv.QUOTE_ALL, encoding='utf-8', mode='a')